In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
CAMINHO_DATA = '/content/drive/MyDrive/Projeto_Futebol/data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3. Modeling and Evaluation (Metodologia CRISP-ML)

## 3.1. Visão Geral e Estratégia de Treino
Nesta fase, construímos, avaliamos e comparamos múltiplos algoritmos de **Machine Learning (IA I e IA II)** e **Deep Learning (TensorFlow/Keras)** para prever o resultado das partidas de futebol (`FTR`: `H`, `D`, `A`).

### Metodologia e Cuidados Principais:
1. **Divisão Temporal (Time-Series Split):** Para evitar *data leakage* (treinar com dados do futuro para prever o passado), utilizamos uma divisão sequencial sem baralhar (*no shuffle*): os **80% de jogos mais antigos** formam o conjunto de treino e os **20% mais recentes** formam o conjunto de teste.
2. **Engenharia de Features para Empates:** Incluem-se variáveis de diferença absoluta entre equipas (`Diff_Forma_L5`, `Diff_GM_L5`, `Diff_GS_L5`) para dar sinalização direta de jogos equilibrados.
3. **Persistência de Artefactos:** Todos os modelos treinados, o escalador (`StandardScaler`) e o codificador (`LabelEncoder`) são guardados automaticamente no diretório `models/`.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Montar Google Drive e definir caminhos
drive.mount('/content/drive', force_remount=True)

CAMINHO_DATA = '/content/drive/MyDrive/Projeto_Futebol/data'
PASTA_MODELS = '/content/drive/MyDrive/Projeto_Futebol/models'
os.makedirs(PASTA_MODELS, exist_ok=True)

# 2. Carregar o dataset preparado
caminho_prep = os.path.join(CAMINHO_DATA, "dataset_futebol_preparado.csv")
df = pd.read_csv(caminho_prep)

# 3. Criar variáveis de diferença (caso ainda não existam no CSV)
if 'Diff_Forma_L5' not in df.columns:
    df['Diff_Forma_L5'] = np.abs(df['Home_Forma_L5'] - df['Away_Forma_L5'])
    df['Diff_GM_L5'] = np.abs(df['Home_Golos_Marcados_L5'] - df['Away_Golos_Marcados_L5'])
    df['Diff_GS_L5'] = np.abs(df['Home_Golos_Sofridos_L5'] - df['Away_Golos_Sofridos_L5'])

features = [
    'Home_Golos_Marcados_L5', 'Away_Golos_Marcados_L5',
    'Home_Golos_Sofridos_L5', 'Away_Golos_Sofridos_L5',
    'Home_Forma_L5', 'Away_Forma_L5',
    'Diff_Forma_L5', 'Diff_GM_L5', 'Diff_GS_L5'
]

X = df[features]
y_raw = df['FTR']

# 4. Encoders e Divisão Temporal (80% Treino / 20% Teste)
le = LabelEncoder()
y = le.fit_transform(y_raw)

corte = int(len(df) * 0.80)
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y[:corte], y[corte:]

# 5. Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Guardar artefactos base
joblib.dump(le, os.path.join(PASTA_MODELS, "label_encoder.joblib"))
joblib.dump(scaler, os.path.join(PASTA_MODELS, "scaler.joblib"))
joblib.dump(features, os.path.join(PASTA_MODELS, "features_list.joblib"))

print(f"✅ Dados Carregados: {len(df)} jogos | Features Preditivas: {len(features)}")
print(f"✅ Conjunto de Treino: {X_train.shape[0]} jogos | Conjunto de Teste: {X_test.shape[0]} jogos")
print(f"✅ Artefactos guardados em: {PASTA_MODELS}")

Mounted at /content/drive
✅ Dados Carregados: 8020 jogos | Features Preditivas: 9
✅ Conjunto de Treino: 6416 jogos | Conjunto de Teste: 1604 jogos
✅ Artefactos guardados em: /content/drive/MyDrive/Projeto_Futebol/models


## 3.2. Otimização de Hiperparâmetros via GridSearchCV (IA I & IA II)
Aplica-se a pesquisa em grelha (`GridSearchCV`) com validação cruzada temporal (`TimeSeriesSplit`) para encontrar a combinação ótima de hiperparâmetros de cada modelo. A métrica de otimização selecionada é o **$F1\text{-Score}$ Macro**, garantindo que o algoritmo não penalize a classe dos empates.

In [14]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

print("=" * 80)
print("🔍 OTIMIZAÇÃO DE HIPERPARÂMETROS (GRIDSEARCHCV)")
print("=" * 80 + "\n")

cv_temporal = TimeSeriesSplit(n_splits=3)

param_grids = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {
            "C": [0.01, 0.1, 1.0, 10.0],
            "class_weight": [None, "balanced"],
            "solver": ["lbfgs", "liblinear"]
        },
        "scaled": True
    },
    "SVM": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            "C": [0.1, 1.0, 10.0],
            "kernel": ["rbf", "linear"],
            "gamma": ["scale", "auto"],
            "class_weight": [None, "balanced"]
        },
        "scaled": True
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [4, 6, 8, 10],
            "min_samples_split": [2, 5, 10],
            "class_weight": [None, "balanced", "balanced_subsample"]
        },
        "scaled": False
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 4, 5],
            "subsample": [0.8, 1.0]
        },
        "scaled": False
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [3, 5, 8, 12],
            "criterion": ["gini", "entropy"],
            "class_weight": [None, "balanced"]
        },
        "scaled": False
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7, 11, 15],
            "weights": ["uniform", "distance"],
            "metric": ["euclidean", "manhattan"]
        },
        "scaled": True
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {
            "var_smoothing": np.logspace(0, -9, num=10)
        },
        "scaled": True
    }
}

resultados_grid = []
modelos_otimizados = {}
previsoes_otimizadas = {}

idx_empate = list(le.classes_).index("D") if "D" in le.classes_ else 1

for nome, config in param_grids.items():
    print(f"⏳ A otimizar {nome}...")

    X_tr = X_train_scaled if config["scaled"] else X_train
    X_te = X_test_scaled if config["scaled"] else X_test

    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        scoring="f1_macro",
        cv=cv_temporal,
        n_jobs=-1,
        verbose=0
    )

    grid.fit(X_tr, y_train)
    melhor_modelo = grid.best_estimator_
    y_pred = melhor_modelo.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)

    f1_classes = f1_score(y_test, y_pred, average=None, zero_division=0)
    f1_empate = f1_classes[idx_empate]

    resultados_grid.append({
        "Modelo": nome,
        "Tipo": "Machine Learning",
        "Accuracy (%)": round(acc * 100, 2),
        "F1-Score Macro (%)": round(f1_macro * 100, 2),
        "Precision Macro (%)": round(prec_macro * 100, 2),
        "Recall Macro (%)": round(rec_macro * 100, 2),
        "F1-Score Empate 'D' (%)": round(f1_empate * 100, 2),
        "Melhores Hiperparâmetros": str(grid.best_params_)
    })

    modelos_otimizados[nome] = melhor_modelo
    previsoes_otimizadas[nome] = y_pred

    nome_ficheiro = nome.lower().replace(" ", "_") + "_opt.joblib"
    joblib.dump(melhor_modelo, os.path.join(PASTA_MODELS, nome_ficheiro))

print("\n✅ Otimização por GridSearchCV concluída!")

🔍 OTIMIZAÇÃO DE HIPERPARÂMETROS (GRIDSEARCHCV)

⏳ A otimizar Logistic Regression...
⏳ A otimizar SVM...
⏳ A otimizar Random Forest...
⏳ A otimizar Gradient Boosting...
⏳ A otimizar Decision Tree...
⏳ A otimizar KNN...
⏳ A otimizar Naive Bayes...

✅ Otimização por GridSearchCV concluída!


## 3.3. Treino do Modelo de Deep Learning em TensorFlow / Keras
Treina-se uma **Rede Neuronal Densa (Multi-Layer Perceptron / ANN)** através da biblioteca **TensorFlow**. Utiliza-se a função de ativação *Softmax* para a camada de saída (3 classes), regularização por *Dropout* e *EarlyStopping* para prevenir o sobreajustamento (*overfitting*).

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

print("⏳ A treinar Rede Neuronal em TensorFlow...")

y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

tf_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

tf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

tf_model.fit(
    X_train_scaled, y_train_cat,
    epochs=100, batch_size=32,
    validation_split=0.2, callbacks=[early_stop], verbose=0
)

y_pred_prob = tf_model.predict(X_test_scaled, verbose=0)
y_pred_tf = np.argmax(y_pred_prob, axis=1)

previsoes_otimizadas["TensorFlow Neural Net"] = y_pred_tf

acc_tf = accuracy_score(y_test, y_pred_tf)
f1_macro_tf = f1_score(y_test, y_pred_tf, average="macro")
prec_macro_tf = precision_score(y_test, y_pred_tf, average="macro", zero_division=0)
rec_macro_tf = recall_score(y_test, y_pred_tf, average="macro", zero_division=0)
f1_empate_tf = f1_score(y_test, y_pred_tf, average=None, zero_division=0)[idx_empate]

resultados_grid.append({
    "Modelo": "TensorFlow Neural Net",
    "Tipo": "Deep Learning",
    "Accuracy (%)": round(acc_tf * 100, 2),
    "F1-Score Macro (%)": round(f1_macro_tf * 100, 2),
    "Precision Macro (%)": round(prec_macro_tf * 100, 2),
    "Recall Macro (%)": round(rec_macro_tf * 100, 2),
    "F1-Score Empate 'D' (%)": round(f1_empate_tf * 100, 2),
    "Melhores Hiperparâmetros": "Dense(64->32->16), Dropout(0.3/0.2), Adam(lr=0.0008)"
})

tf_model.save(os.path.join(PASTA_MODELS, "tensorflow_mlp_opt.keras"))
print("✅ TensorFlow Neural Net concluído!")

⏳ A treinar Rede Neuronal em TensorFlow...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ TensorFlow Neural Net concluído!


## 3.4. Quadro Comparativo e Visualização dos Resultados
Exibe-se a tabela final com os desempenhos ordenados pelo **$F1\text{-Score}$ Macro** (métrica principal para avaliar o equilíbrio entre empates, vitórias em casa e fora) e geram-se a **Matriz de Confusão** e o gráfico de **Importância das Variáveis**.

In [16]:
# 1. Tabela Resumo Ordenada
df_resultados_exp = pd.DataFrame(resultados_grid).sort_values(by="F1-Score Macro (%)", ascending=False).reset_index(drop=True)

print("=" * 110)
print("🏆 QUADRO COMPARATIVO EXPANDIDO DOS MODELOS OTIMIZADOS (GRIDSEARCHCV)")
print("=" * 110)

colunas_visiveis = [
    "Modelo", "Tipo", "Accuracy (%)", "F1-Score Macro (%)",
    "Precision Macro (%)", "Recall Macro (%)", "F1-Score Empate 'D' (%)"
]

print(df_resultados_exp[colunas_visiveis].to_string(index=False))

print("\n" + "=" * 110)
print("⚙️ HIPERPARÂMETROS ÓTIMOS ENCONTRADOS PELO GRIDSEARCHCV:")
print("=" * 110)
for idx, row in df_resultados_exp.iterrows():
    print(f"📌 {row['Modelo']:<22} | {row['Melhores Hiperparâmetros']}")

🏆 QUADRO COMPARATIVO EXPANDIDO DOS MODELOS OTIMIZADOS (GRIDSEARCHCV)
               Modelo             Tipo  Accuracy (%)  F1-Score Macro (%)  Precision Macro (%)  Recall Macro (%)  F1-Score Empate 'D' (%)
                  SVM Machine Learning         46.82               44.34                44.51             44.25                    27.96
  Logistic Regression Machine Learning         46.95               44.11                44.07             44.18                    26.47
        Random Forest Machine Learning         45.01               43.02                43.36             42.91                    27.44
        Decision Tree Machine Learning         43.39               41.52                42.27             41.72                    31.21
    Gradient Boosting Machine Learning         46.32               38.49                41.36             40.32                    16.88
          Naive Bayes Machine Learning         49.50               38.15                41.98             42.

## 3.5. Verificação de Persistência e Exportação de Metadados
Garante-se a correta gravação de todos os artefactos (modelos `.joblib`, rede neuronal `.keras`, escalador e encoders) na pasta `models/`. Cria-se um ficheiro de manifesto (`models_manifest.json`) que regista o ranking dos modelos e identifica o modelo campeão para ser consumido automaticamente pelo **Sistema Multiagente Generativo** e pelo script principal (`main.py`).

In [17]:
import json
from datetime import datetime

melhor_linha = df_resultados_exp.iloc[0]
melhor_modelo_nome = melhor_linha["Modelo"]
melhor_modelo_ficheiro = (
    "tensorflow_mlp_opt.keras"
    if melhor_modelo_nome == "TensorFlow Neural Net"
    else melhor_modelo_nome.lower().replace(" ", "_") + "_opt.joblib"
)

metadados = {
    "data_treino": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "melhor_modelo": {
        "nome": melhor_modelo_nome,
        "ficheiro": melhor_modelo_ficheiro,
        "accuracy": float(melhor_linha["Accuracy (%)"]),
        "f1_score_macro": float(melhor_linha["F1-Score Macro (%)"])
    },
    "classes": list(le.classes_),
    "features": features,
    "resumo_modelos": df_resultados_exp.to_dict(orient="records")
}

caminho_manifesto = os.path.join(PASTA_MODELS, "models_manifest.json")
with open(caminho_manifesto, "w", encoding="utf-8") as f:
    json.dump(metadados, f, indent=4, ensure_ascii=False)

print(f"✅ Manifesto atualizado guardado em: {caminho_manifesto}")
print(f"🏆 Modelo Campeão Selecionado: {melhor_modelo_nome} ({melhor_modelo_ficheiro})")

✅ Manifesto atualizado guardado em: /content/drive/MyDrive/Projeto_Futebol/models/models_manifest.json
🏆 Modelo Campeão Selecionado: SVM (svm_opt.joblib)
